# Mode publish notebook

This notebook expects a Mode SQL query named `BOM Capacity Raw`.
Its final output is `final_df`, which you can add to the report or publish as a local Python dataset in Mode.

In [ ]:
import pandas as pd

FINAL_COLUMN_ORDER = [
    "PATH",
    "PATH_WITHOUT_REVISION",
    "PART_NUMBER",
    "REVISION",
    "PRODUCT_NAME",
    "PRODUCTION_STATE",
    "INDENT_LEVEL",
    "UOM",
    "TRACKING",
    "IS_CONSUMABLE_STORABLE",
    "PARENT_BOM",
    "PARENT_REVISION",
    "TOP_LEVEL_BOM",
    "TOP_LEVEL_REVISION",
    "QUANTITY",
    "PROCUREMENT_INTENT",
    "ADJUSTED_QUANTITY",
    "total rolled up quantity",
    "ADJUSTED_PROCUREMENT_INTENT",
    "GLOBAL_ALTERNATE_PART_NUMBERS",
    "SUBSTITUTE_PART_NUMBERS",
    "Supply Plan and On-Hand Alternates",
    "Latest PO",
    "Latest Supplier",
    "Latest PO Creation Date",
    "Product",
    "Variant",
    "System",
    "Subsystem",
    "Gross Demand for BOM Line in past 8 weeks",
    "Net Total Demand for Part Number in past 8 weeks",
    "Current Week Realized Demand Consumption (each)",
    "Gross Demand for BOM Line in next 8 weeks",
    "Net Total Demand for Part Number in next 8 weeks",
    "Quantity Received, 8-Week Rolling Average (each)",
    "Quantity Received, Average Since Last Receipt (each)",
    "Current On-Hand Quantity (each)",
    "On Hand Delta to Current Week Demand (each)",
    "Current Quarantine Quantity (each)",
    "Current Week Realized Supply (each)",
    "Total Supply Plan, next 8 weeks (each)",
    "Average Supply Plan, next 8 weeks (each)",
    "Quantity Received, 8-Week Rolling Average (each) with alternates",
    "Quantity Received, Average Since Last Receipt (each) with alternates",
    "Current On-Hand Quantity (each) with alternates",
    "on hand product sets including alternates",
    "Current Quarantine Quantity (each) with alternates",
    "Current Week Realized Supply (each) with alternates",
    "Total Supply Plan, next 8 weeks (each) with alternates",
    "Average Supply Plan, next 8 weeks (each) with alternates",
    "On Hand Inventory Status",
    "Supply Plan Status",
    "Avg Shipments Status",
]

EACH_COLUMNS = [
    "Current Week Realized Demand Consumption",
    "Quantity Received, 8-Week Rolling Average",
    "Quantity Received, Average Since Last Receipt",
    "Current On-Hand Quantity",
    "Current Quarantine Quantity",
    "Current Week Realized Supply",
    "Total Supply Plan, next 8 weeks",
    "Average Supply Plan, next 8 weeks",
]

ZERO_FILL_COLUMNS = [
    "Gross Demand for BOM Line in past 8 weeks",
    "Net Total Demand for Part Number in past 8 weeks",
    "Current Week Realized Demand Consumption (each)",
    "Gross Demand for BOM Line in next 8 weeks",
    "Net Total Demand for Part Number in next 8 weeks",
    "Quantity Received, 8-Week Rolling Average (each)",
    "Quantity Received, Average Since Last Receipt (each)",
    "Current On-Hand Quantity (each)",
    "On Hand Delta to Current Week Demand (each)",
    "Current Quarantine Quantity (each)",
    "Current Week Realized Supply (each)",
    "Total Supply Plan, next 8 weeks (each)",
    "Average Supply Plan, next 8 weeks (each)",
    "Quantity Received, 8-Week Rolling Average (each) with alternates",
    "Quantity Received, Average Since Last Receipt (each) with alternates",
    "Current On-Hand Quantity (each) with alternates",
    "Current Quarantine Quantity (each) with alternates",
    "Current Week Realized Supply (each) with alternates",
    "Total Supply Plan, next 8 weeks (each) with alternates",
    "Average Supply Plan, next 8 weeks (each) with alternates",
]

ROLLUP_COLUMN = "total rolled up quantity"
PRODUCT_SETS_COLUMN = "on hand product sets including alternates"
MULTIPLE_ROLLUP_VALUE = "mutliple"

def format_mixed_string_value(value):
    if pd.isna(value):
        return None

    numeric_value = pd.to_numeric(value, errors="coerce")
    if pd.notna(numeric_value) and float(numeric_value).is_integer():
        return str(int(numeric_value))

    return str(value)


def coerce_mixed_output_columns_to_string(df):
    output_columns = []
    for position, column in enumerate(df.columns):
        series = df.iloc[:, position]
        if column == ROLLUP_COLUMN:
            series = series.map(format_mixed_string_value).astype(object)
        output_columns.append(series)

    coerced = pd.concat(output_columns, axis=1)
    coerced.columns = df.columns
    coerced.index = df.index
    return coerced


try:
    import notebooksalamode
except ImportError:
    notebooksalamode = None

if notebooksalamode is not None:
    _original_publish_to_helix = getattr(
        notebooksalamode.publish_to_helix,
        "_ctb_original_publish_to_helix",
        notebooksalamode.publish_to_helix,
    )

    def _ctb_publish_to_helix(dataframe, parquet=False):
        if isinstance(dataframe, pd.DataFrame):
            dataframe = coerce_mixed_output_columns_to_string(dataframe)
        return _original_publish_to_helix(dataframe, parquet=parquet)

    _ctb_publish_to_helix._ctb_original_publish_to_helix = _original_publish_to_helix
    notebooksalamode.publish_to_helix = _ctb_publish_to_helix

try:
    raw_df = datasets["BOM Capacity Raw"].copy()
except Exception:
    raw_df = datasets[0].copy()

final_df = raw_df.copy()

if ROLLUP_COLUMN not in final_df.columns:
    required_rollup_columns = {"PART_NUMBER", "TOP_LEVEL_BOM", "ADJUSTED_QUANTITY"}
    if required_rollup_columns.issubset(final_df.columns):
        top_level_revision = (
            final_df["TOP_LEVEL_REVISION"].fillna("")
            if "TOP_LEVEL_REVISION" in final_df.columns
            else pd.Series("", index=final_df.index)
        )
        rollup_input = pd.DataFrame(
            {
                "PART_NUMBER": final_df["PART_NUMBER"],
                "TOP_LEVEL_BOM": final_df["TOP_LEVEL_BOM"],
                "TOP_LEVEL_REVISION": top_level_revision,
                "ADJUSTED_QUANTITY": pd.to_numeric(final_df["ADJUSTED_QUANTITY"], errors="coerce").fillna(0),
            }
        ).dropna(subset=["PART_NUMBER", "TOP_LEVEL_BOM"])

        if not rollup_input.empty:
            top_level_totals = (
                rollup_input.groupby(["PART_NUMBER", "TOP_LEVEL_BOM", "TOP_LEVEL_REVISION"], dropna=False)[
                    "ADJUSTED_QUANTITY"
                ]
                .sum()
                .reset_index()
            )
            part_totals = (
                top_level_totals.groupby("PART_NUMBER")["ADJUSTED_QUANTITY"]
                .agg(unique_total_count="nunique", rolled_up_quantity="max")
                .reset_index()
            )
            part_totals[ROLLUP_COLUMN] = part_totals.apply(
                lambda row: MULTIPLE_ROLLUP_VALUE
                if row["unique_total_count"] > 1
                else row["rolled_up_quantity"],
                axis=1,
            )
            final_df = final_df.merge(part_totals[["PART_NUMBER", ROLLUP_COLUMN]], on="PART_NUMBER", how="left")

for column in EACH_COLUMNS:
    each_column = f"{column} (each)"
    final_df[each_column] = pd.to_numeric(final_df.get(column), errors="coerce")
    if column in final_df.columns:
        final_df = final_df.drop(columns=[column])

    alternate_column = f"{column} with alternates"
    alternate_each_column = f"{each_column} with alternates"
    final_df[alternate_each_column] = pd.to_numeric(final_df.get(alternate_column), errors="coerce")
    if alternate_column in final_df.columns:
        final_df = final_df.drop(columns=[alternate_column])

if PRODUCT_SETS_COLUMN not in final_df.columns and ROLLUP_COLUMN in final_df.columns:
    on_hand_column = "Current On-Hand Quantity (each) with alternates"
    if on_hand_column in final_df.columns:
        rollup_quantity = pd.to_numeric(final_df[ROLLUP_COLUMN], errors="coerce")
        on_hand_quantity = pd.to_numeric(final_df[on_hand_column], errors="coerce").fillna(0)
        valid_rollup = (
            final_df[ROLLUP_COLUMN].ne(MULTIPLE_ROLLUP_VALUE)
            & rollup_quantity.notna()
            & rollup_quantity.ne(0)
        )
        final_df[PRODUCT_SETS_COLUMN] = pd.Series(pd.NA, index=final_df.index, dtype="Float64")
        final_df.loc[valid_rollup, PRODUCT_SETS_COLUMN] = (
            on_hand_quantity.loc[valid_rollup] / rollup_quantity.loc[valid_rollup]
        )
        final_df[PRODUCT_SETS_COLUMN] = pd.to_numeric(final_df[PRODUCT_SETS_COLUMN], errors="coerce")

if "Commodity" in final_df.columns:
    final_df = final_df.drop(columns=["Commodity"])

for column in FINAL_COLUMN_ORDER:
    if column not in final_df.columns:
        final_df[column] = None

for column in ZERO_FILL_COLUMNS:
    if column in final_df.columns:
        final_df[column] = pd.to_numeric(final_df[column], errors="coerce").fillna(0)

final_df = final_df.loc[:, FINAL_COLUMN_ORDER]
final_df = coerce_mixed_output_columns_to_string(final_df)

In [ ]:
final_df